# 02 — IL Training (with capacity-violation penalty)

Builds the `TransformerClusterer`, then runs `train_il()`, which adds a differentiable capacity-overflow penalty on top of the cross-entropy imitation loss (see `src/losses.py`) so the model is explicitly pushed away from predictions that would overflow a ULD's weight or volume limit — not just implicitly taught via the GreedyLabeller's labels.  
**Run `01_setup.ipynb` first** so that `DATA_DIR`, `SAVE_DIR`, `TRAIN_DIR`, `TEST_DIR`, `TRAIN_META`, `TEST_META`, `IL_SAVE`, `IL_LOG` are defined.

In [ ]:
# Re-run setup (or %run 01_setup.ipynb if variables aren't in scope)
import sys, os
sys.path.insert(0, '..')

import torch

from src.config import (
    DEVICE, N_EPOCHS, BATCH_SIZE, LR, PATIENCE,
    LAMBDA_WEIGHT_PENALTY, LAMBDA_VOLUME_PENALTY,
)
from src.model import TransformerClusterer
from src.labeller import GreedyLabeller
from src.train_il import train_il

# These must be set — either run 01_setup.ipynb or define them here
# DATA_DIR    = '../good_data'
# SAVE_DIR    = '../clustering_v2'
# IL_SAVE     = os.path.join(SAVE_DIR, 'transformer_imitation_v2.pt')
# IL_LOG      = os.path.join(SAVE_DIR, 'il_training_log.csv')
# TRAIN_DIR   = os.path.join(DATA_DIR, 'synthetic_train')
# TEST_DIR    = os.path.join(DATA_DIR, 'synthetic_test')
# TRAIN_META  = os.path.join(TRAIN_DIR, 'metadata.csv')
# TEST_META   = os.path.join(TEST_DIR,  'metadata.csv')

In [ ]:
model = TransformerClusterer().to(DEVICE)

history = train_il(
    model,
    train_dir       = TRAIN_DIR,
    test_dir        = TEST_DIR,
    train_meta_path = TRAIN_META,
    test_meta_path  = TEST_META,
    labeller        = GreedyLabeller(),   # swap to retrain on a different heuristic
    n_epochs        = N_EPOCHS,
    batch_size      = BATCH_SIZE,
    lr              = LR,
    patience        = PATIENCE,
    save_path       = IL_SAVE,
    log_path        = IL_LOG,
    device          = DEVICE,
    lambda_weight   = LAMBDA_WEIGHT_PENALTY,
    lambda_volume   = LAMBDA_VOLUME_PENALTY,
)